59 mins 30.6 secs

## Load libraries

In [15]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm
from collections import defaultdict

## Config

In [16]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV
ROLLING_TRAIN_WINDOW = 120   # 10 years
ROLLING_VAL_WINDOW   = 12    # 1 year

# ---- feature lists (adjust to match your data) ----
continuous_cols = [
    #"AverageNeighbourPrice",
    #"local_I",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    #"LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
]

categorical_cols = [
    "LMIQuadrant__2",
    "LMIQuadrant__3",
    "LMIQuadrant__4",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

# ---- lag plan (always 1 & 12; variants with 2–6; optional 24) ----
lag_combinations = [
    [1, 12],
    [1, 2, 12],
    [1, 2, 3, 12],
    [1, 2, 3, 4, 5, 6, 12],
    [1, 12, 24],
    [1, 2, 12, 24],
    [1, 2, 3, 12, 24],
    [1, 2, 3, 4, 5, 6, 12, 24],
]
all_lags = sorted({l for combo in lag_combinations for l in combo})

# GB tuning grid
arx_param_grid = {
    "alpha": [0.0, 0.1, 1.0, 10.0, 100.0],  # 0.0 = OLS
}

## Metrics

In [17]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

## Load data

In [18]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL])

mask_tv = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
df_tv = df.loc[mask_tv].copy()
df_tv["LA_ID"] = df_tv[ENTITY_COL].astype("category")

# all unique months in train+val window
dates = sorted(df_tv[TIME_COL].unique())

## Training with STL and rolling CV

In [19]:
# =========================================================
# CV STRUCTURE:
#   outer loop   = folds (STL + all lags computed once per fold)
#   mid loop     = lag_set (subselect lag columns, scale, etc.)
#   inner loop   = RF params (fit, predict, metrics)
# =========================================================

# metrics_store[(lag_tuple, params_key)] = dict of metric lists across folds
metrics_store = {}

# Pre-build list of folds based on dates
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW
while True:
    train_end_idx = start_idx
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > len(dates):
        break

    train_start = dates[train_end_idx - ROLLING_TRAIN_WINDOW]
    train_end   = dates[train_end_idx - 1]
    val_start   = dates[val_start_idx]
    val_end     = dates[val_end_idx - 1]

    fold_specs.append((train_start, train_end, val_start, val_end))
    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")

# =========================================================
# MAIN LOOP: FOLDS → (lag_set → ARX params)
# =========================================================
for fold_no, (train_start, train_end, val_start, val_end) in enumerate(fold_specs, start=1):
    print(f"\n=== Fold {fold_no}: "
          f"Train {train_start:%Y-%m}–{train_end:%Y-%m}, "
          f"Val {val_start:%Y-%m}–{val_end:%Y-%m} ===")

    # ---- slice fold train/val ----
    mask_train = (df_tv[TIME_COL] >= train_start) & (df_tv[TIME_COL] <= train_end)
    mask_val   = (df_tv[TIME_COL] >= val_start)   & (df_tv[TIME_COL] <= val_end)

    fold_train = df_tv.loc[mask_train].copy()
    fold_val   = df_tv.loc[mask_val].copy()

    # ---- combine train+val and create TARGET lags once per fold ----
    fold_train["is_train"] = True
    fold_val["is_train"]   = False

    combined = pd.concat([fold_train, fold_val], axis=0)
    combined = combined.sort_values([ENTITY_COL, TIME_COL])

    # create lag columns for ALL lags (superset) on the TARGET
    for lag in all_lags:
        col = f"{TARGET_COL}_lag{lag}"
        combined[col] = combined.groupby(ENTITY_COL)[TARGET_COL].shift(lag)

     # ---- LA fixed effects (one-hot) on combined so train/val share same cols ----
    la_dummies = pd.get_dummies(
        combined[ENTITY_COL].astype("category"),
        prefix="LA",
        drop_first=True,  # avoid perfect collinearity
    )
    combined = pd.concat([combined, la_dummies], axis=1)
    la_cols = list(la_dummies.columns)

    # ---------------------------------------------------------
    # For this fold: loop over lag_set, then ARX params
    # Using the precomputed lag columns in `combined`
    # ---------------------------------------------------------
    for lag_set in lag_combinations:
        print(f"  Lag set: {lag_set}")
        lag_cols = [f"{TARGET_COL}_lag{lag}" for lag in lag_set]

        # split back into train/val
        fold_train_lag = combined[combined["is_train"]].copy()
        fold_val_lag   = combined[~combined["is_train"]].copy()

        # require all chosen lags present
        fold_train_lag = fold_train_lag.dropna(subset=lag_cols)
        fold_val_lag   = fold_val_lag.dropna(subset=lag_cols)

        feature_cols = continuous_cols + categorical_cols + la_cols + lag_cols
        fold_train_lag = fold_train_lag.dropna(subset=feature_cols)
        fold_val_lag   = fold_val_lag.dropna(subset=feature_cols)

        if fold_train_lag.empty or fold_val_lag.empty:
            print("    (skip: no data after lag drop)")
            continue

        # scale continuous + lag features ON TRAIN ONLY (not categoricals / LA dummies)
        scale_cols = continuous_cols + lag_cols
        scaler = StandardScaler()
        fold_train_lag[scale_cols] = scaler.fit_transform(fold_train_lag[scale_cols])
        fold_val_lag[scale_cols]   = scaler.transform(fold_val_lag[scale_cols])

        feature_cols = continuous_cols + categorical_cols + la_cols + lag_cols

        X_train = fold_train_lag[feature_cols]
        y_train = fold_train_lag[TARGET_COL].values

        X_val   = fold_val_lag[feature_cols]
        y_val   = fold_val_lag[TARGET_COL].values

        # inner loop over ARX (Ridge) hyperparameters
        for params in ParameterGrid(arx_param_grid):
            print(f"    ARX params: {params}")
            lag_key = tuple(lag_set)
            params_key = tuple(sorted(params.items()))
            key = (lag_key, params_key)

            if key not in metrics_store:
                metrics_store[key] = {
                    "lag_set": lag_key,
                    "params": params,
                    "mae": [],
                    "rmse": [],
                    "smape": [],
                    "mase": [],
                    "folds": 0,
                }

            model = Ridge(
                alpha=params["alpha"],
                fit_intercept=True,
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            metrics_store[key]["mae"].append(mae(y_val, y_pred))
            metrics_store[key]["rmse"].append(rmse(y_val, y_pred))
            metrics_store[key]["smape"].append(smape(y_val, y_pred))
            metrics_store[key]["mase"].append(mase(y_val, y_pred, y_train))
            metrics_store[key]["folds"] += 1



Number of folds: 5

=== Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03 ===
  Lag set: [1, 12]
    ARX params: {'alpha': 0.0}
    ARX params: {'alpha': 0.1}
    ARX params: {'alpha': 1.0}
    ARX params: {'alpha': 10.0}
    ARX params: {'alpha': 100.0}
  Lag set: [1, 2, 12]
    ARX params: {'alpha': 0.0}
    ARX params: {'alpha': 0.1}
    ARX params: {'alpha': 1.0}
    ARX params: {'alpha': 10.0}
    ARX params: {'alpha': 100.0}
  Lag set: [1, 2, 3, 12]
    ARX params: {'alpha': 0.0}
    ARX params: {'alpha': 0.1}
    ARX params: {'alpha': 1.0}
    ARX params: {'alpha': 10.0}
    ARX params: {'alpha': 100.0}
  Lag set: [1, 2, 3, 4, 5, 6, 12]
    ARX params: {'alpha': 0.0}
    ARX params: {'alpha': 0.1}
    ARX params: {'alpha': 1.0}
    ARX params: {'alpha': 10.0}
    ARX params: {'alpha': 100.0}
  Lag set: [1, 12, 24]
    ARX params: {'alpha': 0.0}
    ARX params: {'alpha': 0.1}
    ARX params: {'alpha': 1.0}
    ARX params: {'alpha': 10.0}
    ARX params: {'alpha': 100.0}
  Lag se

## Results

In [21]:
rows = []
for key, val in metrics_store.items():
    if val["folds"] == 0:
        continue
    rows.append({
        "model_type": "ARX_Ridge",
        "lag_set": val["lag_set"],
        "params": val["params"],
        "folds": val["folds"],
        "MAE_mean":   float(np.mean(val["mae"])),
        "MAE_std":    float(np.std(val["mae"])),
        "RMSE_mean":  float(np.mean(val["rmse"])),
        "RMSE_std":   float(np.std(val["rmse"])),
        "sMAPE_mean": float(np.mean(val["smape"])),
        "sMAPE_std":  float(np.std(val["smape"])),
        "MASE_mean":  float(np.mean(val["mase"])),
        "MASE_std":   float(np.std(val["mase"])),
    })

results_df = pd.DataFrame(rows).sort_values("RMSE_mean").reset_index(drop=True)
print(results_df.head(20))
results_df.to_csv("../../results/arx_leakfree_rollingcv_results.csv", index=False)

   model_type                     lag_set          params  folds  \
0   ARX_Ridge  (1, 2, 3, 4, 5, 6, 12, 24)  {'alpha': 0.0}      5   
1   ARX_Ridge                 (1, 12, 24)  {'alpha': 0.0}      5   
2   ARX_Ridge           (1, 2, 3, 12, 24)  {'alpha': 0.0}      5   
3   ARX_Ridge                  (1, 2, 12)  {'alpha': 0.0}      5   
4   ARX_Ridge              (1, 2, 12, 24)  {'alpha': 0.0}      5   
5   ARX_Ridge      (1, 2, 3, 4, 5, 6, 12)  {'alpha': 0.0}      5   
6   ARX_Ridge                     (1, 12)  {'alpha': 0.0}      5   
7   ARX_Ridge               (1, 2, 3, 12)  {'alpha': 0.0}      5   
8   ARX_Ridge                     (1, 12)  {'alpha': 0.1}      5   
9   ARX_Ridge                 (1, 12, 24)  {'alpha': 0.1}      5   
10  ARX_Ridge                  (1, 2, 12)  {'alpha': 0.1}      5   
11  ARX_Ridge               (1, 2, 3, 12)  {'alpha': 0.1}      5   
12  ARX_Ridge      (1, 2, 3, 4, 5, 6, 12)  {'alpha': 0.1}      5   
13  ARX_Ridge              (1, 2, 12, 24)  {'alp